# Level 1b — Reading the Paper: Forming and Testing Hypotheses

*CAJAL Neuromics 2026 · Computational Mini-Project C10 · Glioblastoma*

In **Level 1** you built a single-nucleus census of two adult glioblastoma (GBM) tumours
*from scratch and without knowing the source paper*: QC → scVI/Harmony integration →
Leiden clustering → cell-type annotation → an inferCNV malignant/TME split → a malignant
**cell-state axis**. You ended with an annotated reference of **117,200 nuclei**.

Now we do something different. **We reveal the paper the data comes from, you read it, and
then you put its claims on trial using *your own* Level 1 results.** This is how real
computational biology works: a paper makes a claim, and you ask *"is that actually what my
data shows?"* — not by re-reading the authors' figure, but by re-deriving it yourself and
seeing whether it holds.

This notebook is **biology-driven and hypothesis-making**. For each claim you will:

1. state the **claim** the paper makes,
2. turn it into a concrete, falsifiable **hypothesis** with an **expected observable**,
3. **test it** with a focused visualization or statistic on your annotated data,
4. **interpret** what you see — does it support, refine, or contradict the paper?

### How this notebook works

We do **no heavy recompute** here — no re-integration, no retraining. Everything runs on the
already-annotated 117k-nucleus object from Level 1, on CPU, in a few minutes. The work is in
the **reasoning and the plots**, not the compute.

**Notation** (same as Level 1):

- 🔬 **TASK:** something to do / a plot to make.
- 💡 **HINT:** a pointer if you're stuck.
- ❓ **QUESTION:** a conceptual question to discuss — think, then look at your plot.
- ⚠️ **CHECKPOINT:** a rough expected range to sanity-check yourself against.

and two new markers specific to Level 1b:

- 🧬 **CLAIM:** a specific statement from the paper we are going to test.
- 🔭 **HYPOTHESIS:** the claim rephrased as a concrete, testable prediction about *your* data.

## 0. Setup

In [ ]:
# --- Setup -----------------------------------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

# Make the project's shared helper package importable (same as Level 1).
sys.path.insert(0, "/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C10/lederer/gbm_space_proj/src")
import gbmspace_utils as gu
from gbmspace_utils.analysis import (
    MALIGNANT_AXIS_MARKERS, MAJOR_CLASS_OF, TME_MARKERS, EMT_MARKERS,
    score_axis, assign_dominant_state,
)

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(5, 4))
%matplotlib inline

print("scanpy", sc.__version__, "| anndata", ad.__version__)

## 1. Revealing the paper 📖

The Level 1 data comes from:

> **A spatiotemporal cancer cell trajectory underlies glioblastoma heterogeneity.**
> Grant de Jong\*, Fani Memi\*, Tannia Gracia\*, Olga Lazareva\*, Oliver Gould, Alexander
> Aivazidis, … Sam Behjati#, Oliver Stegle#, Omer Ali Bayraktar#.
> *bioRxiv* 2025.05.13.653495 (2025). Companion portal: **gbmspace.org**.

🔬 **TASK 1.1 — Read the paper first.** Before running anything below, read at least the
**Abstract** and the first two Results sections (*"Deep spatial multi-omic profiling of GB"*
and *"Malignant cells vary from developmental-like to gliosis and hypoxia states"*), plus skim
**Figure 1**. The whole point of this notebook is to compare *your* data to *their* claims, so
you need their claims in your head first.

Here are the five key findings we will put on trial (all checkable from snRNA-seq alone — we
leave the *spatial* claims for Level 2):

1. **Malignant cells occupy a defined set of transcriptional states**, hierarchically grouped
   into **4 major classes / 9 subclasses**, spanning **developmental-like** programs
   (OPC-NPC-like, NPC-neuronal-like) through a **reactive-astrocyte → gliosis → hypoxia**
   spectrum — *not* one undifferentiated malignant identity.
2. These states occur at **unequal relative abundances** (neuronal-like states are the
   *rarest*) and are **molecularly conserved across tumours** — the same states recur in
   every tumour, not tumour-private inventions.
3. There is a distinct **Proliferative** state, but proliferation is **not confined to it**:
   the cycling cells co-express *either* dev-like *or* gliosis-hypoxia programs — i.e.
   *"both dev-like and gliosis-hypoxia states proliferate in GBs."*
4. The gliosis/hypoxia states (historically called **"mesenchymal-like"**) are a **glial
   injury + hypoxia response, not a classical EMT** — canonical EMT regulators
   (SNAI1/2, TWIST1/2, ZEB1/2) are *"negligible and non-specific."*
5. The **TME is myeloid-dominated**: *"Myeloid cells are the most abundant TME cell type in
   GB"*, with OPCs/oligodendrocytes also abundant and **lymphocytes rare** — an
   immunosuppressive, immunologically "cold" microenvironment.

Each section below tests one of these.

In [ ]:
# Your code for: Load your Level 1 annotated reference


In [ ]:
# Your code for: Recap the Level 1 labels we will reason about


In [ ]:
# Your code for: Fix a biologically meaningful ORDER for the state axis (dev-like -> gliosis -> hypoxia,


In [ ]:
# Your code for: Orient yourself: where do the malignant classes sit on the UMAP you built in Level 1?


## 2. Claim 1 — Malignant cells occupy defined neurodevelopmental + injury states

🧬 **CLAIM (paper).** *"Malignant cells vary from developmental-like to gliosis and hypoxia
states."* The authors hierarchically group malignant clusters into **4 major classes and 9
subclasses**: two **developmental-like** classes (OPC-NPC-like, with OPC markers PDGFRA/OLIG1/SOX6;
and NPC-neuronal-like, with MYT1L/STMN2), an **AC-progenitor → gliosis → hypoxia** class (core
astrocyte markers SLC1A3/GFAP, terminating in HILPDA/VEGFA hypoxia), and a **Proliferative** class.

❓ **QUESTION.** If malignant cells were just "one aggressive blob", every state you labelled in
Level 1 would express the *same* genes. If instead they occupy *distinct* transcriptional
states, each labelled state should be **enriched for its own signature and depleted for the
others**. Which do you expect from a tumour the paper calls "heterogeneous"?

🔭 **HYPOTHESIS.** Scoring each nucleus against the 9 state signatures
(`MALIGNANT_AXIS_MARKERS`) and averaging per assigned state will produce a **diagonal-dominant**
matrix: each state scores highest on *its own* signature. Marker genes will likewise be
**state-specific** on a dotplot.

🔬 **TASK 2.1.** Dotplot the state-defining marker genes across `malignant_state` (use the
`STATE_ORDER` above). 💡 **HINT:** collect a couple of hallmark genes per state so the dotplot
stays readable — e.g. PDGFRA/OLIG1 (OPC), MYT1L/STMN2 (NPC-neuronal), GFAP/AQP4 (AC),
SERPINE1/VEGFA (gliosis/hypoxia), HILPDA (hypoxia), MKI67/TOP2A (proliferative).

In [ ]:
# Your code for: TASK 2.1 -- marker dotplot across malignant states


🔬 **TASK 2.2.** Rather than eyeballing marker genes, **score** each nucleus against every state
signature with the paper's own method (`sc.tl.score_genes`, wrapped here as `score_axis`), then
average the scores per assigned state. Plot the resulting **state × signature** matrix as a
heatmap. 💡 **HINT:** `score_axis(mal, MALIGNANT_AXIS_MARKERS)` returns a
(cells × signatures) DataFrame. A strong diagonal = states are genuinely distinct.

In [ ]:
# Your code for: TASK 2.2 -- state-signature score heatmap


🔬 **TASK 2.3.** Confirm the states are also *distinct genes*, not just distinct scores. Run
`sc.tl.rank_genes_groups` between the malignant **major classes** and look at the top markers of
each. 💡 **HINT:** `method="wilcoxon"`, `groupby="malignant_class"`.

In [ ]:
# Your code for: TASK 2.3 -- differential expression between major classes


## 3. Claim 2 — Unequal abundances, conserved across tumours

🧬 **CLAIM (paper).** The 9 subclasses were *"each observed across multiple tumours"* and the
trajectory *"manifests in a molecularly conserved manner across tumours."* Abundances are
**unequal**: the neuronal-like states are explicitly *"the least abundant states, consistent
with the low frequency of neuronal-like 'Neural' GB tumour subtypes."*

❓ **QUESTION.** Two competing pictures: (a) each tumour invents its *own* private states
(→ composition bar charts would look totally different per donor/site), or (b) the *same* menu
of states recurs everywhere, only the *proportions* shift (→ every donor shows every state, at
different heights). Which does the paper claim, and which does your data show?

🔭 **HYPOTHESIS.** A stacked-bar composition of `malignant_class` **per donor** and **per site**
will show *all* classes present in *both* donors and *all* sites (conservation), with
**NPC-neuronal-like clearly the smallest slice everywhere** (unequal abundance).

🔬 **TASK 3.1.** Build a stacked-bar of malignant-class fractions per **donor** and per
**site**. 💡 **HINT:** `pd.crosstab(..., normalize="index")` gives per-group fractions; a
`.plot(kind="bar", stacked=True)` renders them.

In [ ]:
# Your code for: TASK 3.1 -- composition per donor and per site


⚠️ **CHECKPOINT.** You should see **all four major classes present in both donors and in every
site**. `AC-gliosis-hypoxia` and `OPC-NPC-like` should be the two big slices; `NPC-neuronal-like`
and `Proliferative` should be small everywhere (each only a few percent). If a class is missing
from one donor entirely, revisit your Level 1 annotation.

🔬 **TASK 3.2.** Quantify "conservation" a little more sharply: does *every* site contain *every*
subclass state? Print a state × site count table and check for structural zeros.

In [ ]:
# Your code for: TASK 3.2 -- is every state seen at every site?


## 4. Claim 3 — Proliferation is a program *layered onto* other states

🧬 **CLAIM (paper).** *"The fourth and final major malignant state represented proliferative
cells. While dominated by a proliferation gene expression signature, these cells also expressed
dev-like or gliosis-hypoxia programmes … suggesting that both dev-like and gliosis-hypoxia states
proliferate in GBs."*

This is subtle and worth pausing on. It says the Proliferative state is **not a separate
lineage** — it's a *cell-cycle program* riding on top of whatever developmental-or-injury
identity a cell already has. So cycling cells should carry a **second** identity.

❓ **QUESTION.** If proliferation were a distinct lineage, Proliferative cells would score high
on the cell-cycle signature and *low* on everything else. If instead it's a layered program,
Proliferative cells should score high on cell cycle **and** split into a dev-like subset and a
gliosis-hypoxia subset. What do you predict?

🔭 **HYPOTHESIS.** (i) A cell-cycle score will be **sharply highest in the Proliferative state**
and low elsewhere. (ii) *Within* the Proliferative cells, the dev-like score and the
gliosis-hypoxia score will be **anti-correlated / bimodal** — individual cycling cells lean one
way or the other, reproducing "both dev-like and gliosis-hypoxia states proliferate."

🔬 **TASK 4.1.** Compute a cell-cycle score with `sc.tl.score_genes_cell_cycle` (standard
Tirosh S/G2M gene lists) and violin-plot the G2M score across `malignant_state`.

In [ ]:
# Your code for: TASK 4.1 -- cell-cycle scoring across states


🔬 **TASK 4.2 — the key test.** Take *only* the Proliferative cells and ask what *second*
identity they carry. Compute a **dev-like** score (mean of the OPC-NPC-like and NPC-neuronal-like
signatures) and a **gliosis-hypoxia** score for every malignant cell, then scatter dev-like vs
gliosis-hypoxia **for the Proliferative subset**. 💡 **HINT:** if the paper is right, the
Proliferative cloud will *spread along both axes* (some high-dev, some high-gliosis) rather than
collapse to a single spot.

In [ ]:
# Your code for: TASK 4.2 -- what second identity do cycling cells carry?


## 5. Claim 4 — "Mesenchymal-like" is glial injury + hypoxia, **not** EMT

🧬 **CLAIM (paper).** The gliosis-like and hypoxic states correspond to the historical Neftel
**"MES1"/"MES2" ("mesenchymal-like")** programs. But the authors argue this is a *misnomer*:
*"while … described as mesenchymal-like … we observed **negligible and non-specific expression of
key EMT regulators (e.g. SNAI1/2, TWIST1/2, ZEB1/2)** in these malignant states … our
nomenclature places these states in the more specific biological context of **glial injury
response and hypoxia**."*

❓ **QUESTION.** If the "mesenchymal" label were literally about epithelial-to-mesenchymal
transition, the master EMT transcription factors (SNAI1/2, TWIST1/2, ZEB1/2) should be
**specifically up** in the gliosis/hypoxia class. If instead it's an injury/hypoxia response, the
EMT-TFs should be **flat and non-specific**, while injury (SERPINE1, VIM, ANXA2) and hypoxia
(HILPDA, VEGFA, BNIP3L) genes should be the ones that light up. Which pattern do you expect?

🔭 **HYPOTHESIS.** On a dotplot across `malignant_class`, the EMT-TF panel will be **near-empty
and unselective** (low fraction expressing, no enrichment in AC-gliosis-hypoxia), whereas the
injury/hypoxia panel will be **strongly and specifically** enriched in AC-gliosis-hypoxia.

🔬 **TASK 5.1.** Dotplot the EMT regulators (`EMT_MARKERS`) *side by side* with an injury/hypoxia
panel, grouped by `malignant_class`. 💡 **HINT:** don't `standard_scale` here — you want to see
the *absolute* fraction of cells expressing the EMT-TFs (it will be tiny).

In [ ]:
# Your code for: TASK 5.1 -- EMT regulators vs injury/hypoxia genes


🔬 **TASK 5.2.** Make it quantitative. For each EMT regulator, print the **fraction of
AC-gliosis-hypoxia cells expressing it** and its **mean expression**, and compare to a real
hypoxia marker (HILPDA). 💡 **HINT:** "expressing" = nonzero in `.layers['counts']`.

In [ ]:
# Your code for: TASK 5.2 -- quantify EMT-TF expression in the gliosis/hypoxia class


## 6. Claim 5 — A myeloid-dominated, lymphocyte-poor microenvironment

🧬 **CLAIM (paper).** *"Myeloid cells are the most abundant TME cell type in GB, underlying a
highly immunosuppressive environment."* In the atlas, *"OPCs, mature oligodendrocytes and myeloid
cells were the most abundant"* TME lineages, while *"Lymphocytes were similarly rare."*

❓ **QUESTION.** GBM is famously an immunologically **"cold"** tumour. If that's true here, the
TME compartment you split off in Level 1 should be dominated by **myeloid** cells (microglia +
macrophages) with abundant **oligodendrocytes/OPCs**, and should contain **very few lymphocytes**
(T/NK/B cells). What would the opposite — a "hot", lymphocyte-rich tumour — look like instead?

🔭 **HYPOTHESIS.** Scoring the TME nuclei against broad lineage panels (`TME_MARKERS`) and
assigning each to its best-matching lineage will show **myeloid + oligodendrocyte/OPC as the
largest compartments** and **lymphocytes as a thin sliver** (low single-digit %).

🔬 **TASK 6.1.** On the **TME** nuclei only, score each broad lineage panel (`TME_MARKERS`),
assign each nucleus its dominant lineage (`assign_dominant_state`), and bar-plot the composition.
💡 **HINT:** `assign_dominant_state` just takes the argmax over the score columns.

In [ ]:
# Your code for: TASK 6.1 -- broad TME lineage composition


🔬 **TASK 6.2.** Sanity-check the assignment with a marker dotplot, and pull out the "cold-tumour"
comparison directly: what fraction of TME nuclei are **myeloid** (microglia + macrophage/monocyte)
versus **lymphocyte**? 💡 **HINT:** dotplot `TME_MARKERS` across `tme_lineage`; then compare
myeloid vs lymphocyte percentages.

In [ ]:
# Your code for: TASK 6.2 -- verify lineages + the myeloid-vs-lymphocyte contrast


⚠️ **CHECKPOINT.** Myeloid cells should be one of the largest TME compartments and lymphocytes
should be a *thin* sliver — you should see a myeloid:lymphocyte ratio of roughly **20:1 or more**.
Note that our automated `Developing_Human_Brain` annotation from Level 1 doesn't resolve rare
adult immune subtypes well, so treat the lymphocyte number as an upper bound if anything.

## 7. Synthesis — the scorecard

🔬 **TASK 7.1.** Pull your five verdicts together in one place. For each claim, record whether
your data **supported**, **refined**, or **contradicted** it, and one number/plot that decided it.

In [ ]:
# Your code for: TASK 7.1 -- assemble the hypothesis scorecard


### What's next

You have now put the paper's core single-cell claims on trial and — where the snRNA-seq data can
speak to them — found them to hold. But the paper's headline is **spatial**: a *"spatiotemporal
cancer cell trajectory"* in which dev-like states sit at the infiltrating edge and gliosis/hypoxia
states occupy the necrotic core, coupled to a regionalised myeloid environment. None of that is
testable from dissociated nuclei alone. **Level 2** brings in the matched Visium sections to test
those spatial claims directly — using this very annotated reference to deconvolve the tissue.